In [2]:
import pandas as pd
import numpy as np
import json
import time
import os
from openai import OpenAI
from dotenv import load_dotenv
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed  # For parallel execution

warnings.filterwarnings('ignore')
load_dotenv()

print("=" * 80)
print("Agent #3: Ensemble Quality Assurance (6 Specialist Agents)")
print("=" * 80)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = os.getenv('LLM_MODEL', 'gpt-4o-mini')

# ============================================================================
# 1. Define 6 Specialist Agent Prompts
# ============================================================================
# Each agent evaluates only within its designated area of expertise.

SPECIALIST_PROMPTS = {
    "cf_alignment": """
    You are a [Data Consistency Auditor]. Your only job is numerical verification.

    SCORING RULES — apply strictly in this order:
    1 point  : Any CF target figure is missing from the report body entirely.
    2 points : At least one CF target figure appears, but one or more figures
               are wrong, rounded differently, or contradict the source data.
    3 points : All key CF target figures appear, but at least one percentage
               change or derived value (e.g. ratio, delta) is miscalculated.
    4 points : All CF target figures and percentage changes are cited correctly;
               minor narrative paraphrase of numbers is acceptable.
    5 points : Every CF target figure, every percentage change, and every
               derived value matches the source data exactly — zero errors.

    HOW TO VERIFY:
    - Extract every number that appears in the report.
    - Cross-check each number against the original CF data provided.
    - A figure is WRONG if it differs by any amount, including rounding.
    - Do NOT give partial credit for "close" numbers — either correct or not.
    - Do NOT let strategic quality influence your score.
    """,

    "logic_flow": """
    You are a [Logical Structure Analyst].
    Disregard numerical accuracy entirely and evaluate only whether
    the paragraph flow and causal reasoning are logically sound.
    Verify whether 'A therefore B' arguments are valid.
    """,

    "actionability": """
    You are a [Field Implementation Consultant].
    Evaluate whether this proposal is executable starting tomorrow,
    or whether it is abstract and impractical.
    Check whether specific responsible parties, deadlines, and methodologies are stated.
    """,

    "business_insight": """
    You are a [Business Strategist].
    Assess whether this is a report that merely lists numbers,
    or whether it contains genuine strategic insight for the firm's survival.
    Verify whether it appropriately reflects the characteristics of the wholesale industry.
    """,

    "completeness": """
    You are a [Compliance Officer].
    Evaluate whether the report includes all 6 required sections
    and whether each section has sufficient substance.
    Focus on structural and formal compliance.
    """,

    "clarity": """
    You are a [Communication Specialist].
    Evaluate whether the report is written in clear, accessible language
    suitable for executive readers.
    Check for excessive jargon or vague expressions.
    """
}

# Evaluation weights
WEIGHTS = {
    "cf_alignment": 0.25,
    "actionability": 0.25,
    "business_insight": 0.20,
    "logic_flow": 0.15,
    "completeness": 0.10,
    "clarity": 0.05
}

# ============================================================================
# 2. Specialist Agent Execution Function
# ============================================================================
def call_specialist_agent(agent_name, report_text, cf_reference: str = ""):
    """
    Calls an individual specialist agent and returns its evaluation.
    cf_reference: stringified dict of original → CF target values (for cf_alignment only).
    """
    system_persona = SPECIALIST_PROMPTS[agent_name]

    # CF Alignment agent receives the ground-truth figures for cross-checking
    reference_block = ""
    if agent_name == "cf_alignment" and cf_reference:
        reference_block = f"""
    [SOURCE DATA — ground-truth CF targets to verify against]
    {cf_reference}
    Verify every number in the report against this reference.
    """

    user_prompt = f"""
    Evaluate the following financial consulting report strictly from your
    specialist perspective [{agent_name}].
    {reference_block}
    [Report Content]
    {report_text}

    [Evaluation Criteria]
    Assign a score on a 1 (worst) to 5 (best) scale
    and provide a one-sentence justification citing specific evidence.

    [Output Format (JSON)]
    {{
        "score": score (integer 1–5),
        "reason": "evaluation justification with specific evidence"
    }}
    """

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_persona},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.2  # Low temperature for objective evaluation
        )
        return agent_name, json.loads(response.choices[0].message.content)
    except Exception as e:
        return agent_name, {"score": 1, "reason": f"Error: {str(e)}"}

# ============================================================================
# 3. Ensemble Evaluator
# ============================================================================
class EnsembleEvaluator:
    def evaluate_report(self, report_row):
        """
        Runs 6 specialist agents in parallel and aggregates results.
        """
        report_content = report_row['report_content']

        # Build CF reference string from original/target columns for cf_alignment agent
        cf_ref_lines = []
        for col in report_row.index:
            if col.startswith('original_') or col.startswith('target_'):
                cf_ref_lines.append(f"  {col}: {report_row[col]}")
        cf_reference = "\n".join(cf_ref_lines) if cf_ref_lines else ""

        results = {}

        # 1. Parallel execution for speed
        with ThreadPoolExecutor(max_workers=6) as executor:
            future_to_agent = {
                executor.submit(
                    call_specialist_agent, name, report_content,
                    cf_reference if name == "cf_alignment" else ""
                ): name
                for name in SPECIALIST_PROMPTS.keys()
            }

            for future in as_completed(future_to_agent):
                agent_name, result = future.result()
                results[agent_name] = result

        # 2. Compute composite score (weighted average)
        total_score = 0
        details = {}

        for name, weight in WEIGHTS.items():
            score = results[name].get('score', 1)
            total_score += score * weight
            details[f"score_{name}"] = score
            details[f"reason_{name}"] = results[name].get('reason', 'N/A')

        # 3. Grade assignment using distribution-based calibration
        # Threshold values reflect relative calibration (top/middle/bottom thirds)
        if total_score >= 3.60:
            decision = "Pass"
        elif total_score >= 3.30:
            decision = "Conditional Pass"
        else:
            decision = "Reject"

        # 4. Return aggregated result
        final_output = {
            'company_id': report_row['company_id'],
            'final_score': round(total_score, 2),
            'decision': decision,
            **details  # Include individual scores and reasons
        }

        return final_output

# ============================================================================
# 4. Execution
# ============================================================================
try:
    reports = pd.read_csv('agent2_consulting_reports.csv')
    # reports = reports.head(3)  # Uncomment for testing with 3 records
except FileNotFoundError:
    raise FileNotFoundError("Step 6 output file not found.")

evaluator = EnsembleEvaluator()
evaluated_reports = []

print(f"\nStarting 6-specialist evaluation for {len(reports)} reports...")
print("Note: Running in parallel.")

try:
    from tqdm import tqdm
    iterator = tqdm(reports.iterrows(), total=len(reports), desc="Evaluating")
except ImportError:
    iterator = reports.iterrows()

for idx, row in iterator:
    eval_result = evaluator.evaluate_report(row)

    full_record = row.to_dict()
    full_record.update(eval_result)
    evaluated_reports.append(full_record)

# ============================================================================
# 5. Save and Display Results
# ============================================================================
df_eval = pd.DataFrame(evaluated_reports)
df_eval.to_csv('agent3_ensemble_results.csv', index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print("Evaluation complete")
print("=" * 80)

# Grade distribution statistics
pass_cnt = (df_eval['decision'] == 'Pass').sum()
cond_cnt = (df_eval['decision'] == 'Conditional Pass').sum()
reject_cnt = (df_eval['decision'] == 'Reject').sum()
avg_score = df_eval['final_score'].mean()

print(f"Total evaluated: {len(df_eval)}")
print(f"- Pass:             {pass_cnt} ({pass_cnt/len(df_eval)*100:.1f}%)")
print(f"- Conditional Pass: {cond_cnt} ({cond_cnt/len(df_eval)*100:.1f}%)")
print(f"- Reject:           {reject_cnt} ({reject_cnt/len(df_eval)*100:.1f}%)")
print(f"- Average score:    {avg_score:.2f}")

# Sample output
if not df_eval.empty:
    sample = df_eval.iloc[0]
    print(f"\n[Sample Evaluation: Company {sample['company_id']}]")
    print(f"Composite Score: {sample['final_score']} ({sample['decision']})")
    print("-" * 40)
    print(f"1. CF Alignment    ({sample['score_cf_alignment']}): {sample['reason_cf_alignment']}")
    print(f"2. Logic & Flow    ({sample['score_logic_flow']}): {sample['reason_logic_flow']}")
    print(f"3. Actionability   ({sample['score_actionability']}): {sample['reason_actionability']}")
    print(f"4. Business Insight({sample['score_business_insight']}): {sample['reason_business_insight']}")
    print(f"5. Completeness    ({sample['score_completeness']}): {sample['reason_completeness']}")
    print(f"6. Clarity         ({sample['score_clarity']}): {sample['reason_clarity']}")

Agent #3: Ensemble Quality Assurance (6 Specialist Agents)

Starting 6-specialist evaluation for 266 reports...
Note: Running in parallel.


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 266/266 [11:46<00:00,  2.65s/it]


Evaluation complete
Total evaluated: 266
- Pass:             0 (0.0%)
- Conditional Pass: 5 (1.9%)
- Reject:           261 (98.1%)
- Average score:    2.97

[Sample Evaluation: Company 75.0]
Composite Score: 2.8 (Reject)
----------------------------------------
1. CF Alignment    (2): The report contains correct CF target figures for bankruptcy probability (99.0% to 46.9%) but miscalculates the percentage change as 52.1 percentage points instead of the correct 52.0 percentage points, and the derived values for increases in liabilities and expenses are also incorrect.
2. Logic & Flow    (3): The report presents a coherent analysis of the company's financial situation and outlines improvement scenarios, but the proposed increases in liabilities and expenses without clear justification for their necessity may lead to confusion regarding the overall strategy for financial recovery.
3. Actionability   (2): The proposal lacks specific responsible parties, deadlines, and clear methodologies 